# ps04_solution.ipynb: AURA Subsystem 04 (Plasma Ignition & Ethical Steering Core)
Mission 4, SEC-04-ORI Betelgeuse. Run the cells top to bottom (Kernel > Restart & Run All). Needs `pip install torch numpy`.

| Role | Module | Section |
|---|---|---|
| Systems Architect | `dpo_loss.py` | 1 |
| Protocol Officer | `reward_engine.py` | 2 |
| Data Specialist | `simpo_loss.py`, `length_bias_diagnostics.py` | 3 |
| Security Sentinel | `activation_steering.py` | 4 |
| Full squad | alignment tax audit, sign-off | 5-6 |

All results below come from a tiny toy model (`toy_lm.py`); they show the mechanics, not real-LLM alignment.

In [1]:
import sys, os, random, copy
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == '04_Systems_Architect' else os.getcwd()
for d in ('01_Data_Specialist','02_Protocol_Specialist','03_Security_Sentinel','04_Systems_Architect'):
    sys.path.insert(0, os.path.join(ROOT, d))
import torch
import toy_lm as T
print('torch', torch.__version__)

torch 2.14.1+cpu


## 1. Systems Architect: DPO loss and Bradley-Terry log-ratios
Sanity check: with identical policy and reference the margin is 0, so the loss must equal ln 2 = 0.6931.

In [2]:
from dpo_loss import dpo_loss, dpo_gradient_weight
z = torch.zeros(4)
print('loss at zero margin:', dpo_loss(z, z, z, z, beta=0.1).loss.item())
pc = torch.tensor([-12.3], requires_grad=True); pr = torch.tensor([-15.1], requires_grad=True)
rc, rr = torch.tensor([-13.0]), torch.tensor([-14.2])
dpo_loss(pc, pr, rc, rr, 0.3).loss.backward()
w = dpo_gradient_weight(pc.detach(), pr.detach(), rc, rr, 0.3)
print('autograd grad(chosen):', pc.grad.item(), ' analytic -w:', -w.item())

loss at zero margin: 0.6931471824645996
autograd grad(chosen): -0.11467563360929489  analytic -w: -0.11467563360929489


## 2. Protocol Officer: implicit reward and pair comparison

In [3]:
from reward_engine import PairComparisonEngine, evaluate_implicit_reward
base = T.pretrain(T.TinyLM(), steps=300)
ref = T.frozen_copy(base)
rng = random.Random(7)
pairs = T.make_pairs(8, True, rng)
def lp(m, seqs): return T.seq_logps(m, seqs)
with torch.no_grad():
    c, _ = lp(base, [p[0] for p in pairs]); r, _ = lp(base, [p[1] for p in pairs])
    rc_, _ = lp(ref, [p[0] for p in pairs]); rr_, _ = lp(ref, [p[1] for p in pairs])
res, summ = PairComparisonEngine(beta=0.1).compare([f'pair-{i}' for i in range(8)], c, r, rc_, rr_)
print(PairComparisonEngine.to_markdown(res, summ))
print('(policy == reference here, so all margins are 0: this is the baseline before alignment training)')

| Prompt | r(chosen) | r(rejected) | Margin | P(chosen) | Correct |
|---|---|---|---|---|---|
| pair-0 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-1 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-2 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-3 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-4 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-5 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-6 | 0.000 | 0.000 | 0.000 | 0.500 | no |
| pair-7 | 0.000 | 0.000 | 0.000 | 0.500 | no |

Pairs: 8, accuracy 0.000, mean margin 0.000, mean P(chosen) 0.500, ties 8.
(policy == reference here, so all margins are 0: this is the baseline before alignment training)


## 3. Data Specialist: DPO vs SimPO and the length-bias diagnostic
Training pairs make the winning answer the long one. The test set mixes long-winner and short-winner pairs. A reward that merely prefers length passes the first kind and fails the second.

In [4]:
import ps04_engine as E
print(E.demo_length_bias())

| Method | Pairs | corr(len diff, margin) | Acc (chosen longer) | Acc (chosen shorter) | Exploit score | Verdict |
|---|---|---|---|---|---|---|
| Reference model (before training) | 200 | 0.668 | 0.750 | 0.120 | 0.630 | LENGTH BIAS DETECTED |
| DPO | 200 | -0.983 | 1.000 | 1.000 | 0.000 | NO STRONG LENGTH BIAS |
| SimPO | 200 | 0.787 | 1.000 | 1.000 | 0.000 | NO STRONG LENGTH BIAS |

- Reference model (before training): overall reward accuracy 0.435
- DPO: overall reward accuracy 1.000
- SimPO: overall reward accuracy 1.000

Training pairs had the chosen (benign) answer LONGER. The test set mixes both cases. 'Exploit score' = accuracy when chosen is longer minus accuracy when chosen is shorter. Toy model, single seed: do not read small differences as meaningful.


## 4. Security Sentinel: contrast vector and activation steering

In [5]:
print(E.demo_steering())

| alpha (negative = steer away from malicious persona) | P(malicious tokens) | P(benign tokens) |
|---|---|---|
| 0.0 | 0.9938 | 0.0055 |
| -2.0 | 0.9793 | 0.0193 |
| -5.0 | 0.6733 | 0.3224 |
| -10.0 | 0.0113 | 0.9881 |
| -20.0 | 0.0000 | 0.9999 |

Context: malicious-persona prompts. Weights unchanged after steering: True. Large |alpha| can push the toy model off-distribution; check that fluency (benign mass) is still sensible.


## 5. Alignment tax audit
Fill `example_scores.json` with your own measured MMLU / GSM8K / HumanEval / refusal scores before and after alignment. The shipped file holds placeholder zeros, so no tax is claimed.

In [6]:
print(E.demo_tax())

Template only (placeholder zeros, no benchmark was run):

| Benchmark | Before | After | Delta | Relative tax % | Verdict |
|---|---|---|---|---|---|
| MMLU | 0.0 | 0.0 | 0.0 | 0.0 | WITHIN TOLERANCE |
| GSM8K | 0.0 | 0.0 | 0.0 | 0.0 | WITHIN TOLERANCE |
| HumanEval | 0.0 | 0.0 | 0.0 | 0.0 | WITHIN TOLERANCE |
| refusal_harmful_rate | 0.0 | 0.0 | 0.0 | 0.0 | WITHIN TOLERANCE |

Mean capability tax: 0.0% (tolerance 2.0%).
